In [80]:
import json
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from scipy.sparse import diags
from sklearn.feature_extraction import text
from tokenizers import ByteLevelBPETokenizer


# Part D — Experiment 1: Inspect the Sparse Representation


## 7.2. Chuẩn bị dữ liệu và biểu diễn TF-IDF

### Raw document


In [81]:
DATA_PATH = "./datasets/c4-train.00000-of-01024-30K.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_documents = [json.loads(line)["text"] for line in f]

## Tokenizer

In [82]:
TOKEN_PATTERN = re.compile(r"[^\W\d_]+", flags=re.UNICODE)

tokenized_documents = [TOKEN_PATTERN.findall(doc.lower()) for doc in raw_documents]

## CountVectorizer

In [83]:
vectorizer = CountVectorizer(analyzer=lambda x: x, lowercase=False)
count_matrix = vectorizer.fit_transform(tokenized_documents)
vocabulary = vectorizer.get_feature_names_out()

## TF IDF TF-IDF matrix

In [84]:
# 1. Tính ma trận TF (L1 norm theo từng hàng)
tf_matrix = normalize(count_matrix, norm="l1", axis=1)

# 2. Tính Document Frequency (DF) và IDF
df = np.array((count_matrix > 0).sum(axis=0)).flatten()
N = count_matrix.shape[0]
idf_vector = np.log10(N / df)

# 3. Tính ma trận TF-IDF
tfidf_matrix = tf_matrix.dot(diags(idf_vector))

## 7.3. Kiểm tra kích thước ma trận

In [85]:
N, V = count_matrix.shape
print(f"Number of documents (N) = {N:,}")
print(f"Vocabulary size (V) = {V:,}")
print(f"Matrix shape = {count_matrix.shape}")

Number of documents (N) = 30,000
Vocabulary size (V) = 177,230
Matrix shape = (30000, 177230)


## 7.4. Kiểm tra sparsity

In [86]:
nnz = count_matrix.nnz
total_elements = N * V 
sparsity = 1.0 - (nnz / total_elements)

print(f"Số lượng phần tử khác không (nnz) = {nnz:,}")
print(f"Tổng số phần tử (N x V) = {total_elements:,}")
print(f"Sparsity (S) = {sparsity:.6f} ({sparsity * 100:.4f}%)")

Số lượng phần tử khác không (nnz) = 4,956,905
Tổng số phần tử (N x V) = 5,316,900,000
Sparsity (S) = 0.999068 (99.9068%)


### **Câu hỏi 7.4:**
**Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều $V$?**

- **Lý do đại số tuyến tính:** Để có thể so sánh khoảng cách hoặc độ tương đồng (như Cosine Similarity, tích vô hướng $\mathbf{x}^\top \mathbf{y}$) giữa các văn bản bất kỳ hoặc giữa văn bản với truy vấn (query), mọi document trong không gian Vector Space Model (VSM) bắt buộc phải nằm trên **cùng một không gian vector đa chiều thống nhất**.
- Không gian này có số chiều bằng kích thước toàn bộ từ vựng $V = 177,230$. Mỗi chiều tương ứng với một từ định danh cụ thể trong vocabulary.
- Một văn bản thực tế chỉ chứa vài chục đến vài trăm từ vựng khác nhau (chiếm $< 0.1\%$ của $V$), do đó các vị trí của những từ không xuất hiện bắt buộc phải nhận giá trị $0$. Điều này dẫn tới tính chất cực kỳ thưa (**sparsity > 99.9%**).

## 7.5. Inspect vocabulary

In [87]:
# 1. 20 terms phổ biến nhất theo document frequency
top20_df_idx = np.argsort(df)[::-1][:20]
df_top20_df = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": vocabulary[top20_df_idx],
    "DF (Document Frequency)": df[top20_df_idx],
    "IDF": np.round(idf_vector[top20_df_idx], 4)
})

# 2. 20 terms có IDF cao nhất (df = 1)
top20_idf_idx = np.argsort(idf_vector)[::-1][:20]
df_top20_idf = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": vocabulary[top20_idf_idx],
    "DF": df[top20_idf_idx],
    "IDF": np.round(idf_vector[top20_idf_idx], 4)
})

# 3. 20 terms có TF-IDF cao nhất trong 1 Document
doc_idx = 0
doc_tfidf = tfidf_matrix.getrow(doc_idx).toarray().flatten()
top20_doc_idx = np.argsort(doc_tfidf)[::-1][:20]
df_top20_doc = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": vocabulary[top20_doc_idx],
    f"TF-IDF in Doc {doc_idx}": np.round(doc_tfidf[top20_doc_idx], 4),
    "DF": df[top20_doc_idx],
    "IDF": np.round(idf_vector[top20_doc_idx], 4)
})


print("20 terms phổ biến nhất theo document frequency:")
display(df_top20_df.head(20))

print("\n20 terms có IDF cao nhất:")
display(df_top20_idf.head(20))
    
print(f"\n20 terms có TF-IDF cao nhất trong Document {doc_idx}:")
display(df_top20_doc.head(20))

20 terms phổ biến nhất theo document frequency:


,Rank,Term,DF (Document Frequency),IDF
0,1,the,27893,0.0316
1,2,and,27423,0.0390
2,3,to,26690,0.0508
3,4,of,26031,0.0616
4,5,a,25943,0.0631
5,6,in,25229,0.0752
6,7,for,23651,0.1033
7,8,is,22740,0.1203
8,9,with,21405,0.1466
9,10,on,20262,0.1704



20 terms có IDF cao nhất:


,Rank,Term,DF,IDF
0,1,aaaa,1,4.4771
1,2,𐌼𐌿𐌽𐌳𐍃,1,4.4771
2,3,ﾺc,1,4.4771
3,4,ﬂuid,1,4.4771
4,5,ﬂoors,1,4.4771
5,6,ﬂexibility,1,4.4771
6,7,ﬁxes,1,4.4771
7,8,ﬁt,1,4.4771
8,9,aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaahhhhhhhh...,1,4.4771
9,10,aaas,1,4.4771



20 terms có TF-IDF cao nhất trong Document 0:


,Rank,Term,TF-IDF in Doc 0,DF,IDF
0,1,bbq,0.0781,87,2.5376
1,2,class,0.0412,1377,1.3382
2,3,balay,0.0344,1,4.4771
3,4,kcbs,0.0321,2,4.1761
4,5,meat,0.0317,260,2.0621
5,6,lonestar,0.0308,3,4.0000
6,7,missoula,0.0267,10,3.4771
7,8,apron,0.0248,18,3.2218
8,9,smoker,0.0246,19,3.1984
9,10,timelines,0.0230,31,2.9858


> ### **Trả lời câu hỏi phân tích 7.5:**
> 
> 1. **Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?**
>    - **Không.** Một từ xuất hiện rất nhiều trong corpus sẽ có Document Frequency ($df$) rất cao, làm cho $IDF = \log_{10}(N / df)$ rất nhỏ (tiến gần về $0$, ví dụ các stopwords như `the`, `and`, `to` có IDF $\approx 0.03 - 0.05$). Do $TF\text{-}IDF = TF \times IDF$, dù tần suất xuất hiện $TF$ trong tài liệu có cao thì giá trị $IDF$ rất nhỏ cũng làm cho tổng thể $TF\text{-}IDF$ của từ này bị giảm mạnh. TF-IDF được thiết kế để phạt các từ xuất hiện phổ biến khắp mọi nơi.
> 
> 2. **Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?**
>    - **Không.** Một từ có IDF cao nghĩa là từ đó rất hiếm trong toàn bộ corpus ($df$ nhỏ, ví dụ $df = 1$). Tuy nhiên, nếu một document cụ thể **không chứa** từ đó thì số lần xuất hiện $c(t, d) = 0 \implies TF = 0 \implies TF\text{-}IDF = 0$. Một từ chỉ có TF-IDF cao trong một văn bản khi và chỉ khi nó vừa có $IDF$ cao (từ mang tính đặc trưng, hiếm) VÀ vừa xuất hiện nhiều lần ($TF$ cao) trong chính văn bản đó (minh chứng rõ ràng ở Doc 0 với các từ như `bbq`, `kcbs`, `balay`, `meat`).

# Part F — Experiment 2: Preprocessing Ablation

## Pipeline A (Minimal)


In [88]:
with open("./datasets/c4-train.00000-of-01024-30K.json", "r", encoding="utf-8") as f:
    raw_documents = [json.loads(line)["text"] for line in f]

def tokenize_pipeline_a(doc):
    # Pipeline A: Lowercase -> White-space tokenization (giữ dấu câu)
    return doc.lower().split()

# 2. Tập queries kiểm thử
eval_queries_ablation = {
    "barbecue cooking class": [0, 984, 1429, 3554, 7015],
    "food safety standards regulations": [1721, 4182, 5299, 8705],
    "natural language processing": [5699, 5912, 8270, 14384, 17437],
    "deep learning healthcare": [287, 430, 454, 644, 1431],
    "heart attack treatment": [799, 1124, 1422, 1605, 2189],
    "transformer language model": [28973]
}

tokenized_corpus = [tokenize_pipeline_a(doc) for doc in raw_documents]
avg_tokens = float(np.mean([len(tokens) for tokens in tokenized_corpus]))

vec = CountVectorizer(analyzer=lambda x: x, lowercase=False)
X = vec.fit_transform(tokenized_corpus)
N_p, V_p = X.shape
sparsity_p = 1.0 - (X.nnz / (N_p * V_p))
vocab_set = set(vec.get_feature_names_out())

# OOV Rate
query_tokens = [t for q in eval_queries_ablation.keys() for t in tokenize_pipeline_a(q)]
oov_count = sum(1 for t in query_tokens if t not in vocab_set)
oov_rate = oov_count / len(query_tokens) if query_tokens else 0.0

# Search Performance (Recall@5 & MRR)
df_p = np.array((X > 0).sum(axis=0)).flatten()
idf_p = np.log10(N_p / np.maximum(df_p, 1))
tf_p = normalize(X, norm="l1", axis=1)
tfidf_p = tf_p.dot(diags(idf_p))
tfidf_p_norm = normalize(tfidf_p, norm="l2", axis=1)

recalls, mrrs = [], []
for q_text, rel_docs in eval_queries_ablation.items():
    q_vec = vec.transform([tokenize_pipeline_a(q_text)])
    q_tf = normalize(q_vec, norm="l1", axis=1)
    q_tfidf = q_tf.dot(diags(idf_p))
    q_norm = normalize(q_tfidf, norm="l2", axis=1)
    scores = (tfidf_p_norm.dot(q_norm.T)).toarray().flatten()

    top5 = np.argsort(scores)[::-1][:5]
    rel_set = set(rel_docs)
    recalls.append(len(set(top5) & rel_set) / len(rel_set))

    ranked_all = np.argsort(scores)[::-1]
    rr = 0.0
    for rank_pos, doc_id in enumerate(ranked_all[:100], 1):
        if doc_id in rel_set:
            rr = 1.0 / rank_pos
            break
    mrrs.append(rr)

avg_recall = np.mean(recalls)
avg_mrr = np.mean(mrrs)

metrics_a = {
    "Pipeline": "Pipeline A (Minimal)",
    "Vocabulary size": f"{V_p:,}",
    "Average tokens/document": f"{avg_tokens:.2f}",
    "Matrix sparsity": f"{sparsity_p * 100:.4f}%",
    "OOV rate": f"{oov_rate * 100:.2f}%",
    "Recall@5": f"{avg_recall * 100:.2f}%",
    "MRR": f"{avg_mrr:.4f}"
}

display(pd.DataFrame([metrics_a]))


,Pipeline,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate,Recall@5,MRR
0,Pipeline A (Minimal),"473,388",361.09,99.9611%,0.00%,6.67%,0.2099


## Pipeline B (Normalized)


In [89]:
with open("./datasets/c4-train.00000-of-01024-30K.json", "r", encoding="utf-8") as f:
    raw_documents = [json.loads(line)["text"] for line in f]


stop_words_set = text.ENGLISH_STOP_WORDS

def tokenize_pipeline_b(doc):
    # Pipeline B: Lowercase -> Loại dấu câu -> Loại Stopwords
    tokens = re.findall(r"\b[a-z]+\b", doc.lower())
    return [t for t in tokens if t not in stop_words_set]

# 2. Tập queries kiểm thử
eval_queries_ablation = {
    "barbecue cooking class": [0, 984, 1429, 3554, 7015],
    "food safety standards regulations": [1721, 4182, 5299, 8705],
    "natural language processing": [5699, 5912, 8270, 14384, 17437],
    "deep learning healthcare": [287, 430, 454, 644, 1431],
    "heart attack treatment": [799, 1124, 1422, 1605, 2189],
    "transformer language model": [28973]
}

tokenized_corpus = [tokenize_pipeline_b(doc) for doc in raw_documents]
avg_tokens = float(np.mean([len(tokens) for tokens in tokenized_corpus]))

vec = CountVectorizer(analyzer=lambda x: x, lowercase=False)
X = vec.fit_transform(tokenized_corpus)
N_p, V_p = X.shape
sparsity_p = 1.0 - (X.nnz / (N_p * V_p))
vocab_set = set(vec.get_feature_names_out())

# OOV Rate
query_tokens = [t for q in eval_queries_ablation.keys() for t in tokenize_pipeline_b(q)]
oov_count = sum(1 for t in query_tokens if t not in vocab_set)
oov_rate = oov_count / len(query_tokens) if query_tokens else 0.0

# Search Performance (Recall@5 & MRR)
df_p = np.array((X > 0).sum(axis=0)).flatten()
idf_p = np.log10(N_p / np.maximum(df_p, 1))
tf_p = normalize(X, norm="l1", axis=1)
tfidf_p = tf_p.dot(diags(idf_p))
tfidf_p_norm = normalize(tfidf_p, norm="l2", axis=1)

recalls, mrrs = [], []
for q_text, rel_docs in eval_queries_ablation.items():
    q_vec = vec.transform([tokenize_pipeline_b(q_text)])
    q_tf = normalize(q_vec, norm="l1", axis=1)
    q_tfidf = q_tf.dot(diags(idf_p))
    q_norm = normalize(q_tfidf, norm="l2", axis=1)
    scores = (tfidf_p_norm.dot(q_norm.T)).toarray().flatten()

    top5 = np.argsort(scores)[::-1][:5]
    rel_set = set(rel_docs)
    recalls.append(len(set(top5) & rel_set) / len(rel_set))

    ranked_all = np.argsort(scores)[::-1]
    rr = 0.0
    for rank_pos, doc_id in enumerate(ranked_all[:100], 1):
        if doc_id in rel_set:
            rr = 1.0 / rank_pos
            break
    mrrs.append(rr)

avg_recall = np.mean(recalls)
avg_mrr = np.mean(mrrs)

metrics_b = {
    "Pipeline": "Pipeline B (Normalized)",
    "Vocabulary size": f"{V_p:,}",
    "Average tokens/document": f"{avg_tokens:.2f}",
    "Matrix sparsity": f"{sparsity_p * 100:.4f}%",
    "OOV rate": f"{oov_rate * 100:.2f}%",
    "Recall@5": f"{avg_recall * 100:.2f}%",
    "MRR": f"{avg_mrr:.4f}"
}

display(pd.DataFrame([metrics_b]))


,Pipeline,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate,Recall@5,MRR
0,Pipeline B (Normalized),"167,107",188.36,99.9286%,0.00%,14.17%,0.2868


## Pipeline C (Subword)


In [90]:
with open("./datasets/c4-train.00000-of-01024-30K.json", "r", encoding="utf-8") as f:
    raw_documents = [json.loads(line)["text"] for line in f]


# Huấn luyện Subword BPE tokenizer trên tập mẫu đại diện cho Pipeline C
sample_texts = raw_documents[:1000]
bpe_tokenizer = ByteLevelBPETokenizer()
bpe_tokenizer.train_from_iterator(sample_texts, vocab_size=10000, min_frequency=2)

def tokenize_pipeline_c(doc):
    # Pipeline C: BPE Subword Tokenization
    return bpe_tokenizer.encode(doc.lower()).tokens

# 2. Tập queries kiểm thử
eval_queries_ablation = {
    "barbecue cooking class": [0, 984, 1429, 3554, 7015],
    "food safety standards regulations": [1721, 4182, 5299, 8705],
    "natural language processing": [5699, 5912, 8270, 14384, 17437],
    "deep learning healthcare": [287, 430, 454, 644, 1431],
    "heart attack treatment": [799, 1124, 1422, 1605, 2189],
    "transformer language model": [28973]
}

tokenized_corpus = [tokenize_pipeline_c(doc) for doc in raw_documents]
avg_tokens = float(np.mean([len(tokens) for tokens in tokenized_corpus]))

vec = CountVectorizer(analyzer=lambda x: x, lowercase=False)
X = vec.fit_transform(tokenized_corpus)
N_p, V_p = X.shape
sparsity_p = 1.0 - (X.nnz / (N_p * V_p))
vocab_set = set(vec.get_feature_names_out())

# OOV Rate
query_tokens = [t for q in eval_queries_ablation.keys() for t in tokenize_pipeline_c(q)]
oov_count = sum(1 for t in query_tokens if t not in vocab_set)
oov_rate = oov_count / len(query_tokens) if query_tokens else 0.0

# Search Performance (Recall@5 & MRR)
df_p = np.array((X > 0).sum(axis=0)).flatten()
idf_p = np.log10(N_p / np.maximum(df_p, 1))
tf_p = normalize(X, norm="l1", axis=1)
tfidf_p = tf_p.dot(diags(idf_p))
tfidf_p_norm = normalize(tfidf_p, norm="l2", axis=1)

recalls, mrrs = [], []
for q_text, rel_docs in eval_queries_ablation.items():
    q_vec = vec.transform([tokenize_pipeline_c(q_text)])
    q_tf = normalize(q_vec, norm="l1", axis=1)
    q_tfidf = q_tf.dot(diags(idf_p))
    q_norm = normalize(q_tfidf, norm="l2", axis=1)
    scores = (tfidf_p_norm.dot(q_norm.T)).toarray().flatten()

    top5 = np.argsort(scores)[::-1][:5]
    rel_set = set(rel_docs)
    recalls.append(len(set(top5) & rel_set) / len(rel_set))

    ranked_all = np.argsort(scores)[::-1]
    rr = 0.0
    for rank_pos, doc_id in enumerate(ranked_all[:100], 1):
        if doc_id in rel_set:
            rr = 1.0 / rank_pos
            break
    mrrs.append(rr)

avg_recall = np.mean(recalls)
avg_mrr = np.mean(mrrs)

metrics_c = {
    "Pipeline": "Pipeline C (Subword)",
    "Vocabulary size": f"{V_p:,}",
    "Average tokens/document": f"{avg_tokens:.2f}",
    "Matrix sparsity": f"{sparsity_p * 100:.4f}%",
    "OOV rate": f"{oov_rate * 100:.2f}%",
    "Recall@5": f"{avg_recall * 100:.2f}%",
    "MRR": f"{avg_mrr:.4f}"
}

display(pd.DataFrame([metrics_c]))


,Pipeline,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate,Recall@5,MRR
0,Pipeline C (Subword),"7,859",548.55,97.0985%,0.00%,4.17%,0.1833


## So sánh các pipeline

Chạy cell import thư viện ở đầu phần 7 trước. Sau đó có thể chạy riêng từng Pipeline A, B, C; chạy đủ ba pipeline trước khi chạy cell so sánh.


In [91]:
metrics_records = [metrics_a, metrics_b, metrics_c]
df_ablation = pd.DataFrame(metrics_records)
print("\nSo sánh")
display(df_ablation)



So sánh


,Pipeline,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate,Recall@5,MRR
0,Pipeline A (Minimal),"473,388",361.09,99.9611%,0.00%,6.67%,0.2099
1,Pipeline B (Normalized),"167,107",188.36,99.9286%,0.00%,14.17%,0.2868
2,Pipeline C (Subword),"7,859",548.55,97.0985%,0.00%,4.17%,0.1833


### **9.5. Câu hỏi phân tích:**

1. **Lowercasing làm thay đổi vocabulary như thế nào?**
   - Lowercasing giúp gộp các biến thể chữ hoa/chữ thường của cùng một từ (ví dụ: `Learning`, `learning`, `LEARNING` trở thành một token duy nhất `learning`).
   - Thao tác này giúp giảm kích thước từ vựng đáng kể, tăng tần suất thống kê thực sự của từ trong văn bản, hạn chế phân mảnh đặc trưng và giảm tỷ lệ OOV.

2. **Stopword removal có luôn cải thiện representation không?**
   - **Không luôn luôn.** Trong tìm kiếm từ khóa theo chủ đề (topic-based search), việc loại bỏ stopword giúp loại bỏ nhiễu và tăng trọng số cho các từ mang nghĩa thực. Tuy nhiên, trong các bài toán phụ thuộc vào cấu trúc ngữ pháp, phủ định (ví dụ: `not good` vs `good`), hay tên riêng/thành ngữ (ví dụ: tên phim `"To Be or Not to Be"` nếu bỏ stopword sẽ hoàn toàn biến mất), stopword removal có thể làm mất thông tin quan trọng.

3. **Việc loại punctuation có thể làm mất thông tin gì?**
   - Dấu câu mang các tín hiệu ngữ nghĩa quan trọng:
   - Cấu trúc số và định lượng: `$1,000`, `3.14`, `100%`.
   - Thuật ngữ kỹ thuật và lập trình: `C++`, `C#`, `node.js`, `U.S.A.`.
   - Phân tách ranh giới câu và cảm xúc: dấu chấm câu kết thúc ý niệm, dấu hỏi `?` thể hiện câu hỏi, dấu ngoặc kép trích dẫn.

4. **Pipeline nào tạo ra sparse matrix nhất?**
   - **Pipeline A (Minimal)** tạo ra ma trận thưa nhất (**99.9611%**), vì từ vựng của nó rất lớn (473,388 từ). Việc giữ lại dấu câu dính liền với từ khiến mỗi từ bị nhân bản thành nhiều biến thể hiếm khác nhau (ví dụ: `learning,`, `learning.`, `"learning"`), làm tăng $V$ cực lớn trong khi mỗi văn bản chỉ kích hoạt một vài biến thể.

5. **Pipeline nào cho search tốt nhất?**
   - **Pipeline B (Normalized)** cho kết quả tìm kiếm tốt nhất (Recall@5 = 14.17%, MRR = 0.2868), vượt trội so với Pipeline A (Recall@5 = 6.67%, MRR = 0.2099) và Pipeline C (Recall@5 = 4.17%, MRR = 0.1833).

6. **Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?**
   - **Không.** Bằng chứng thực nghiệm rõ ràng: Pipeline C (Subword) có kích thước từ vựng nhỏ nhất ($V = 7,859$), ma trận ít thưa nhất ($97.0985\%$), nhưng lại cho hiệu năng tìm kiếm **thấp nhất** ($Recall@5 = 4.17\%$, $MRR = 0.1833$). Lý do là việc băm nhỏ từ thành các mẩu subword unigram làm mất đi tính toàn vẹn của khái niệm và sinh ra rất nhiều false positives (các tài liệu không liên quan nhưng có chung tiền tố/hậu tố ngắn).